# ЛР 02.2 — Локальный разбор ошибок и сегментов (TODO)

## Цель
- взять лучшую пару `model + feature_set` из ЛР 01;
- разобрать наиболее уверенные false positive и false negative;
- построить локальные объяснения ошибок;
- дополнить локальный анализ сегментной сводкой по ошибкам.

In [ ]:
# Что делаем: Подключаем библиотеки и настраиваем рабочее окружение ноутбука.
# Зачем: Без корректных импортов и констант следующие шаги не будут воспроизводимыми.
# Как читать результат: После выполнения этой ячейки не должно быть ошибок импорта; переменные окружения должны появиться в памяти.
# Типичные ошибки: Частая ошибка — запускать следующие ячейки до инициализации импортов и путей к данным.

# Подключаем зависимости для этого шага.
from pathlib import Path
import importlib.util

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

cwd = Path.cwd().resolve()
candidates = [
    cwd,
    cwd.parent,
    cwd / '02-model-interpretability-and-explainability',
    cwd.parent / '02-model-interpretability-and-explainability',
]
BASE_DIR = next((path for path in candidates if (path / 'lab_utils.py').exists()), None)
if BASE_DIR is None:
    raise FileNotFoundError('Не удалось найти lab_utils.py. Откройте ноутбук из папки модуля 02 или из корня репозитория.')
spec = importlib.util.spec_from_file_location('lab02_utils', BASE_DIR / 'lab_utils.py')
lab = importlib.util.module_from_spec(spec)
spec.loader.exec_module(lab)

SEED = lab.SEED
OUTPUT_DIR = BASE_DIR / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 120)

## Шаг 1. Выбор лучшей пары `model + feature_set` из ЛР 01

Переход к следующему шагу: зафиксируйте выводы текущего шага и используйте их как вход следующего блока.


In [ ]:
# Что делаем: Загружаем входные данные и артефакты предыдущих шагов.
# Зачем: Этот шаг задает исходный контекст: без него метрики и графики будут считаться по неверным данным.
# Как читать результат: Проверьте размеры таблиц и названия ключевых колонок: это главный индикатор корректной загрузки.
# Типичные ошибки: Частая ошибка — использовать не тот файл или устаревший артефакт из другой лабораторной работы.

# Читаем данные и артефакты, с которыми будем работать дальше.
datasets = lab.load_course_datasets()
feature_sets = lab.load_feature_sets()
model_results = lab.load_lab01_model_results()

model_factories = {
    'LogisticRegression': lambda: LogisticRegression(max_iter=3000, class_weight='balanced', random_state=SEED),
    'RandomForest': lambda: RandomForestClassifier(
        n_estimators=400,
        min_samples_leaf=3,
        class_weight='balanced',
        random_state=SEED,
        n_jobs=-1,
    ),
    'LinearSVC': lambda: LinearSVC(C=1.0, class_weight='balanced', random_state=SEED),
}

prepared = {}
selection_rows = []
# Итерируемся по объектам и последовательно накапливаем результаты.
for dataset_name, df in datasets.items():
    x, y = lab.split_xy(df)
    x_train, x_test, y_train, y_test = lab.train_test_split_stratified(x, y)
    best_config = lab.choose_best_model_config(model_results, feature_sets, dataset_name)
    prepared[dataset_name] = {
        'x_train': x_train,
        'x_test': x_test,
        'y_train': y_train,
        'y_test': y_test,
        'best_config': best_config,
        'selected_features': feature_sets[dataset_name][best_config['feature_set']],
    }
    selection_rows.append(best_config)

selection_summary = pd.DataFrame(selection_rows)
selection_summary

## Шаг 2. Обучение выбранных моделей и локальные объяснения ошибок

Переход к следующему шагу: зафиксируйте выводы текущего шага и используйте их как вход следующего блока.


In [ ]:
# Что делаем: Выполняем очередной вычислительный блок текущего шага лабораторной работы.
# Зачем: Этот блок готовит промежуточный результат, который используется в следующей ячейке.
# Как читать результат: После выполнения проверьте вывод и убедитесь, что значения выглядят реалистично.
# Типичные ошибки: Частая ошибка — переходить дальше без проверки промежуточного результата.

artifacts = {}
error_frames = []
segment_frames = []

# Итерируемся по объектам и последовательно накапливаем результаты.
for dataset_name, ctx in prepared.items():
    best_config = ctx['best_config']
    model_name = best_config['model']
    artifact = lab.fit_selected_model(
        ctx['x_train'],
        ctx['x_test'],
        ctx['y_train'],
        ctx['y_test'],
        ctx['selected_features'],
        model_factories[model_name](),
    )
    artifacts[dataset_name] = artifact

    error_frames.append(
        lab.build_error_case_explanations(
            artifact=artifact,
            dataset_name=dataset_name,
            model_name=model_name,
            feature_set_name=best_config['feature_set'],
        )
    )
    segment_frames.append(lab.build_default_segment_tables(artifact, dataset_name))

error_case_explanations = pd.concat(error_frames, ignore_index=True)
segment_error_summary = pd.concat(segment_frames, ignore_index=True)
error_case_explanations.head(20)

## Шаг 3. Сегментный взгляд на ошибки

Переход к следующему шагу: зафиксируйте выводы текущего шага и используйте их как вход следующего блока.


In [ ]:
# Что делаем: Выполняем очередной вычислительный блок текущего шага лабораторной работы.
# Зачем: Этот блок готовит промежуточный результат, который используется в следующей ячейке.
# Как читать результат: После выполнения проверьте вывод и убедитесь, что значения выглядят реалистично.
# Типичные ошибки: Частая ошибка — переходить дальше без проверки промежуточного результата.

segment_error_summary.sort_values(['dataset', 'error_rate'], ascending=[True, False]).head(20)

## Самостоятельное изучение по ходу работы

### Что изучено в этом шаге
- TODO (обязательно): 3-5 предложений о том, чем локальное объяснение отличается от глобального.
- TODO (обязательно): поясните, почему один и тот же кейс можно объяснить разными способами.

### Сравнение с альтернативами
- TODO (обязательно): сравните perturbation-анализ с линейными contribution или с сегментным анализом.
- Формат: когда один подход полезнее другого и почему.

### Источники
- TODO (обязательно): минимум 1-2 источника (URL и/или `study-notes/*.md`).

### Глоссарий незнакомых терминов (обязательно)
- TODO (обязательно): добавьте минимум 2-3 новых термина в `study-notes/glossary.md`.
- В конце блока напишите строку: `Глоссарий обновлен: <термин_1>, <термин_2>, ...`.

## Контрольные точки
1. В `error_case_explanations` есть обе группы ошибок: `false_positive` и `false_negative`.
2. В `segment_error_summary` есть как минимум 2 признака сегментации на датасет.
3. Локальные объяснения не пустые и отсортированы по `importance_value`.

## Обязательные самостоятельные задания (без образца в solutions)

### Задание 1. Сравнение false positive и false negative
- Возьмите `error_case_explanations`.
- Сравните, какие признаки чаще появляются в объяснениях двух типов ошибок.
- Отдельно прокомментируйте `medical` и `finance`.

### Задание 2. Проверка сегментов риска
- Возьмите `segment_error_summary`.
- Найдите сегменты с наибольшим `error_rate`, `false_positive_rate` и `false_negative_rate`.
- Сопоставьте выводы сегментного анализа с локальными объяснениями отдельных кейсов.

### Задание 3. Экспорт артефакта и краткий decision memo
- Сохраните `outputs/error_case_explanations.csv`.
- Напишите 3-5 предложений: где более простая модель предпочтительнее даже при близких метриках.

In [ ]:
# Что делаем: Выполняем очередной вычислительный блок текущего шага лабораторной работы.
# Зачем: Этот блок готовит промежуточный результат, который используется в следующей ячейке.
# Как читать результат: После выполнения проверьте вывод и убедитесь, что значения выглядят реалистично.
# Типичные ошибки: Частая ошибка — переходить дальше без проверки промежуточного результата.

# TODO(обязательно):
# 1) Проверьте required columns для error_case_explanations.
# 2) Сохраните DataFrame в outputs/error_case_explanations.csv.
# 3) Добавьте краткий decision memo в markdown-блок выше.

raise NotImplementedError(
    'Самостоятельный блок не завершен: сохраните outputs/error_case_explanations.csv и заполните выводы по ошибкам.'
)
